In [0]:
import pyspark.sql.functions as F




In [0]:
input_path = 's3://ims-dbr-raw/fhv_trip_records/'
output_path = 's3://ims-dbr-stage/fhv_trip_records/'

In [0]:
df = spark.read.parquet(input_path)
display(df)

#Schema

In [0]:
df_stage = df \
.withColumn('dispatching_base_num', F.col('dispatching_base_num').cast('string')) \
.withColumn('pickup_datetime', F.col('pickup_datetime').cast('timestamp')) \
.withColumnRenamed('dropOff_datetime', 'dropoff_datetime') \
.withColumn('dropoff_datetime', F.col('dropoff_datetime').cast('timestamp')) \
.withColumnRenamed('PUlocationID', 'PULocationID') \
.withColumn('PULocationID', F.col('PULocationID').cast('int')) \
.withColumnRenamed('DOlocationID', 'DOLocationID') \
.withColumn('DOLocationID', F.col('DOLocationID').cast('int')) \
.withColumn('SR_Flag', F.col('SR_Flag').cast('int')) \
.withColumn('Affiliated_base_number', F.col('Affiliated_base_number').cast('String')) \
.withColumn('trip_duration', F.timestamp_diff('minute', F.col('pickup_datetime'), F.col('dropoff_datetime'))) \
.withColumn('pu_hour_of_day', F.hour('pickup_datetime')) \
.withColumn('pu_day_of_week', F.dayofweek('pickup_datetime')) \
.withColumn('pu_year', F.year('pickup_datetime')) \
.withColumn('pu_month', F.month('pickup_datetime')) \
.withColumn('do_hour_of_day', F.hour('dropoff_datetime')) \
.withColumn('do_day_of_week', F.dayofweek('dropoff_datetime')) \
.withColumn('do_year', F.year('dropoff_datetime')) \
.withColumn('do_month', F.month('dropoff_datetime'))\
\
.withColumn('is_pu_unknown', (F.col('PULocationID') == 264) | (F.col('PULocationID') == 265)) \
.withColumn('is_do_unknown', (F.col('DOLocationID') == 264) | (F.col('DOLocationID') == 265)) \
.withColumn('is_dispatching_base_unknown', F.col('dispatching_base_num').isNull()) \
.withColumn('is_pu_unknown', (F.col('PULocationID') == 264) | (F.col('PULocationID') == 265)) \
.withColumn('is_do_unknown', (F.col('DOLocationID') == 264) | (F.col('DOLocationID') == 265)) \
.fillna({
    'PULocationID': 265,
    'DOLocationID': 265,
    'SR_Flag': 0}) \
.dropDuplicates(subset = ['dispatching_base_num', 'pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID'])



#Writing to Silver

In [0]:
## Write to Stage Table
df_stage.write \
  .mode("overwrite") \
  .parquet(output_path)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS nytaxi.staging.fhv_trip_records
USING PARQUET
LOCATION 's3://ims-dbr-stage/fhv_trip_records/';